# Streamlit Inference Check

This notebook validates the saved inference artifacts for:
- price prediction
- market segmentation

It reuses the same inference modules that power the Streamlit app.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.config import load_project_configs
from src.utils.paths import find_project_root

root = find_project_root(project_root)
configs = load_project_configs(root)

root, list(configs.keys())

(WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation'),
 ['main_config',
  'paths_config',
  'features_config',
  'model_registry',
  'regression_config',
  'clustering_config',
  'streamlit_config'])

In [2]:
from src.inference.predict_price import predict_price_from_dict
from src.inference.predict_cluster import predict_cluster_from_dict
from streamlit_app.utils.load_models import load_regression_artifacts, load_clustering_artifacts

In [3]:
sample_payload = {
    "carat": 1.05,
    "cut": "Ideal",
    "color": "G",
    "clarity": "VS2",
    "depth": 61.8,
    "table": 57.0,
    "x": 6.5,
    "y": 6.4,
    "z": 4.0,
}
sample_payload

{'carat': 1.05,
 'cut': 'Ideal',
 'color': 'G',
 'clarity': 'VS2',
 'depth': 61.8,
 'table': 57.0,
 'x': 6.5,
 'y': 6.4,
 'z': 4.0}

In [4]:
reg_artifacts = load_regression_artifacts(root, configs)
reg_artifacts["artifact_path"]

WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/artifacts/regression/best_regression_model.pkl')

In [5]:
price_result = predict_price_from_dict(
    payload=sample_payload,
    configs=configs,
    model_bundle=reg_artifacts,
)
price_result

{'predicted_price_usd': 6378.48,
 'predicted_price_inr': 598242.47,
 'used_exchange_rate': 93.7908,
 'price_band': 'Luxury',
 'engineered_preview': {'volume': 166.4,
  'volume_proxy': 166.4,
  'length_width_ratio': 1.015625,
  'depth_pct_from_dimensions': 62.01550387596899,
  'carat_squared': 1.1025}}

In [6]:
cluster_artifacts = load_clustering_artifacts(root, configs)
cluster_artifacts["artifact_path"]

WindowsPath('F:/DATA SCIENCE/Projects/Diamond Dynamics Price Prediction and Market Segmentation/artifacts/clustering/best_clustering_model.pkl')

In [7]:
cluster_result = predict_cluster_from_dict(
    payload=sample_payload,
    configs=configs,
    cluster_bundle=cluster_artifacts,
)
cluster_result

{'cluster': -1,
 'cluster_name': 'Large Premium Segment - Fair',
 'model_name': 'dbscan'}

# Streamlit App Notes

## Purpose
This application exposes two capabilities from the project:
1. diamond price prediction
2. diamond market segmentation

## Pages
- `1_Price_Prediction.py`
- `2_Market_Segmentation.py`

## Input Features
Raw user inputs:
- carat
- cut
- color
- clarity
- depth
- table
- x
- y
- z

## Inference Flow
### Price prediction
1. validate raw inputs
2. build one-row dataframe
3. add runtime engineered features
4. load saved regression artifact
5. generate price in USD
6. convert result to INR
7. display price card

### Market segmentation
1. validate raw inputs
2. build one-row dataframe
3. add runtime engineered features
4. load saved clustering bundle
5. subset to clustering feature columns
6. run saved preprocessor
7. predict cluster
8. map cluster to saved cluster name

## Important artifact fallback logic
The app first checks config-defined artifact paths.
If they do not exist, it falls back to:
- `artifacts/regression/best_regression_model.pkl`
- `artifacts/clustering/best_clustering_model.pkl`

## Manual test scenarios
- ideal cut, mid-carat diamond
- low-carat budget diamond
- larger premium diamond
- invalid zero carat
- invalid x/y/z dimensions

## Screenshots to capture later
- home page
- price prediction result
- cluster prediction result